#### THIRD ATTEMPT

## Data Processing

### OpTc Dataset
Overview
The OpTC (Operationally Transparent Cyber) dataset was developed by Five Directions, under the DARPA Transparent Computing programme, to support research into large-scale cyber-security monitoring and attack detection. It contains endpoint telemetry collected from Windows 10 computers, recording system and network activity through eCAR (extended Cyber Analytics Repository) events. This dataset contains records, including things such as processes, files, network flows, registry activity and other host events. The original release contains roughly a terabyte of compressed data from hundreds of hosts. It includes benign activity as well as red-team attack activity.

I will be using the corrected 2026 version of the OpTC dataset. INRIA reports that the original dataset contains errors involving unique identifiers and other event properties and recommends using the corrected version instead. It is available at: https://entrepot.recherche.data.gouv.fr/dataset.xhtml?persistentId=doi%3A10.57745%2FUXCWOC&utm_source=chatgpt.com

**Project goal:**

The goal of this project is to investigate the use of ML/DL to predict the occurrence of cybersecurity attacks. The project will first analyse and preprocess the OpTC event data to identify behavioural patterns associated with malicious activity, before transforming the sequential telemetry into suitable features and time-based samples for modelling. Different ML/DL approaches will then be evaluated to determine whether patterns in system and network activity can provide sufficient information to identify or predict an impending cyber attack.

**Chapter goal:**

In this chapter, I loaded the files of interest in JSON format and converted them to Parquet format for easier reloading and processing. To keep the dataset manageable, I used only a subset of the OpTC dataset. This subset includes data from the 16th, when only benign activity occurred. For this day, I downloaded telemetry from an arbitrary group of 25 computers (51–75). I also downloaded data from the 23rd–25th, which represents the evaluation period. For these days, I included telemetry from both attack hosts and computers where only benign activity occurred. This was done to prevent the model from simply associating the later dates with malicious activity rather than learning the actual patterns associated with attacks. The computers whose telemetry was downloaded include:

| Date | Computer Group/Hosts | Activity |
|------|----------------------|----------|
| 16 September 2019 | 51–75 | Benign |
| 23 September 2019 | 201| Attack Host |
| 23 September 2019 | 202-204| Benign  |
| 24 September 2019 | 501, 811 | Attack Hosts |
| 24 September 2019 | 502, 503, 812, 813 | Benign |
| 25 September 2019 | 51, 351 | Attack Hosts |
| 25 September 2019 | 52, 53, 352, 353 | Benign |

In [1]:
# Imports
import json
import pandas as pd
from pathlib import Path



#### 1. Extract and convert target .json files to .parquet

In [2]:

# ============================================================
# 1. Set input and output locations
# ============================================================

raw_files = Path("../../../datasets/OpTC_data/raw")
processed_files = Path("../../../datasets/OpTC_data/processed")


# ============================================================
# 2. Process the full benign group from 16 September
# ============================================================

benign_group = (
    "2019-09-16",
    "AIA-51-75"
)

date, group = benign_group

folder = raw_files / date / group

print(f"\nProcessing benign group: {date} — {group}")
print(f"Folder: {folder}")

files = list(folder.glob("*.json"))

print(f"Found {len(files)} files")


for file in files:

    print(f"Reading: {file.name}")

    df = pd.read_json(
        file,
        lines=True
    )

    output_name = f"{date}_{group}_{file.stem}.parquet"
    output_path = processed_files / output_name

    df.to_parquet(
        output_path,
        engine="pyarrow",
        index=False
    )

    print(f"Saved → {output_path.name}")


# ============================================================
# 3. Select attacked hosts + control hosts from attack days
# ============================================================

selected_hosts = [

    # --------------------------------------------------------
    # Evaluation Day 1 — 23 September
    # --------------------------------------------------------

    # Attacked host
    ("2019-09-23", "AIA-201-225", "sysclient0201"),

    # Control / comparison hosts
    ("2019-09-23", "AIA-201-225", "sysclient0202"),
    ("2019-09-23", "AIA-201-225", "sysclient0203"),
    ("2019-09-23", "AIA-201-225", "sysclient0204"),


    # --------------------------------------------------------
    # Evaluation Day 2 — 24 September
    # --------------------------------------------------------

    # Attacked hosts
    ("2019-09-24", "AIA-501-525", "sysclient0501"),
    ("2019-09-24", "AIA-801-825", "sysclient0811"),

    # Control / comparison hosts
    ("2019-09-24", "AIA-501-525", "sysclient0502"),
    ("2019-09-24", "AIA-501-525", "sysclient0503"),
    ("2019-09-24", "AIA-801-825", "sysclient0812"),
    ("2019-09-24", "AIA-801-825", "sysclient0813"),


    # --------------------------------------------------------
    # Evaluation Day 3 — 25 September
    # --------------------------------------------------------

    # Attacked hosts
    ("2019-09-25", "AIA-51-75", "sysclient0051"),
    ("2019-09-25", "AIA-351-375", "sysclient0351"),

    # Control / comparison hosts
    ("2019-09-25", "AIA-51-75", "sysclient0052"),
    ("2019-09-25", "AIA-51-75", "sysclient0053"),
    ("2019-09-25", "AIA-351-375", "sysclient0352"),
    ("2019-09-25", "AIA-351-375", "sysclient0353")
]


# ============================================================
# 4. Convert selected attack-day hosts to Parquet
# ============================================================

for date, group, host in selected_hosts:

    folder = raw_files / date / group

    print(f"\nProcessing: {date} — {group} — {host}")
    print(f"Folder: {folder}")

    # Find only the JSON file corresponding to this host
    files = list(folder.glob(f"*{host}.json"))

    print(f"Found {len(files)} matching file(s)")

    if len(files) == 0:
        print(f"No file found for {host}")
        continue

    for file in files:

        print(f"Reading: {file.name}")

        df = pd.read_json(
            file,
            lines=True
        )

        output_name = f"{date}_{group}_{host}.parquet"
        output_path = processed_files / output_name

        df.to_parquet(
            output_path,
            engine="pyarrow",
            index=False
        )

        print(f"Saved → {output_path.name}")


print("\nFinished converting selected OpTC telemetry.")



Processing benign group: 2019-09-16 — AIA-51-75
Folder: ../../../datasets/OpTC_data/raw/2019-09-16/AIA-51-75
Found 25 files
Reading: AIA-51-75.ecar-2019-09-16-sysclient0073.json
Saved → 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0073.parquet
Reading: AIA-51-75.ecar-2019-09-16-sysclient0065.json
Saved → 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0065.parquet
Reading: AIA-51-75.ecar-2019-09-16-sysclient0053.json
Saved → 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0053.parquet
Reading: AIA-51-75.ecar-2019-09-16-sysclient0069.json
Saved → 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0069.parquet
Reading: AIA-51-75.ecar-2019-09-16-sysclient0068.json
Saved → 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0068.parquet
Reading: AIA-51-75.ecar-2019-09-16-sysclient0052.json
Saved → 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0052.parquet
Reading: AIA-51-75.ecar-2019-09-16-sysclient0064.json
Saved → 2019-09-16_AIA-51-75_AIA-51